# Generalist Gaming Agent
Attach the **Generalist Gaming Agent** competition data and turn Internet off.
Save & Run All. The normal run writes a placeholder; Kaggle scores the submission
only during its competition rerun. The CPU policy needs no model attachments.

In [ ]:
import os
import sys
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
DATA_ROOT = None  # override with Path("/kaggle/input/...") if autodetection fails
MODEL_DIR = None  # optional: Path("/kaggle/input/.../model-weights")

if DATA_ROOT is None:
    candidates = (
        INPUT / "competitions/generalist-gaming-agent",
        INPUT / "generalist-gaming-agent",
        *(p.parent.parent for pattern in ("*/gga/protocol.py", "*/*/gga/protocol.py")
          for p in INPUT.glob(pattern)),
    )
else:
    candidates = (Path(DATA_ROOT),)

DATA_ROOT = next((p for p in candidates if (p / "gga/protocol.py").is_file()
                  and (p / "kaggle_evaluation/gga/gga_inference_server.py").is_file()), None)
if DATA_ROOT is None:
    raise FileNotFoundError("Attach the Generalist Gaming Agent competition data, or set DATA_ROOT.")

WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
sys.path[:0] = [str(WORK), str(DATA_ROOT)]
os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, (
    str(WORK), str(DATA_ROOT), os.environ.get("PYTHONPATH", ""),
)))
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")

import numpy
from PIL import Image

In [ ]:
# Generated from the repository sources by build_notebook.py.
SOURCES = {
    'agent.py': (
        'from gga.protocol import Action, Observation, ToolSpec\n'
        '\n'
        'from gaming_agent.policy import Policy\n'
        '\n'
        '\n'
        'class Agent:\n'
        '    def __init__(self):\n'
        '        self.policy = Policy()\n'
        '\n'
        '    def step(self, obs: Observation, tools: list[ToolSpec]) -> Action:\n'
        '        tool, params = self.policy.step(obs, tools)\n'
        '        return Action(tool=tool, params=params)\n'
    ),
    'gaming_agent/__init__.py': (
        '"""Offline generalist gaming policy and synthetic benchmark."""\n'
    ),
    'gaming_agent/model.py': (
        'import base64\n'
        'import json\n'
        'import os\n'
        'from queue import Empty, SimpleQueue\n'
        'from threading import Thread\n'
        'from urllib.request import ProxyHandler, Request, build_opener\n'
        '\n'
        '\n'
        'class LocalModel:\n'
        '    def __init__(self, port, model, timeout=2.0):\n'
        '        self.endpoint = f"http://127.0.0.1:{port}/v1/chat/completions"\n'
        '        self.model = model\n'
        '        self.timeout = timeout\n'
        '        self.results = SimpleQueue()\n'
        '        self.running = False\n'
        '\n'
        '    @classmethod\n'
        '    def from_env(cls):\n'
        '        model = os.environ.get("GGA_MODEL_NAME")\n'
        '        if not model:\n'
        '            return None\n'
        '        try:\n'
        '            port = int(os.environ.get("GGA_MODEL_PORT", "8000"))\n'
        '        except ValueError:\n'
        '            return None\n'
        '        return cls(port, model) if 1 <= port <= 65535 else None\n'
        '\n'
        '    def _request(self, game, turn, brief, jpeg, schemas):\n'
        '        proposal = None\n'
        '        try:\n'
        '            data = base64.b64encode(jpeg).decode("ascii")\n'
        '            prompt = (f"Play this browser game. {brief[:3000]}\\n"\n'
        '                      f"Available input tools: {json.dumps(schemas)[:10000]}\\n"\n'
        '                      \'Return only a JSON object {"tool": "name", "params": {}}. \'\n'
        '                      "Use coordinates in screenshot pixels. Use skip when unsure.")\n'
        '            payload = {"model": self.model, "temperature": 0, "max_tokens": 128, "messages": [\n'
        '                {"role": "user", "content": [{"type": "text", "text": prompt}, {\n'
        '                    "type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{data}"},\n'
        '                }]},\n'
        '            ]}\n'
        '            opener = build_opener(ProxyHandler({}))\n'
        '            request = Request(self.endpoint, data=json.dumps(payload).encode(),\n'
        '                              headers={"Content-Type": "application/json"})\n'
        '            with opener.open(request, timeout=self.timeout) as response:\n'
        '                content = json.loads(response.read(65536))["choices"][0]["message"]["content"]\n'
        '                proposal = json.loads(content)\n'
        '        except (OSError, ValueError, LookupError, TypeError):\n'
        '            pass\n'
        '        finally:\n'
        '            self.results.put((game, turn, proposal))\n'
        '\n'
        '    def poll(self, game, turn, brief, jpeg, schemas):\n'
        '        proposal = None\n'
        '        try:\n'
        '            previous_game, previous_turn, result = self.results.get_nowait()\n'
        '        except Empty:\n'
        '            pass\n'
        '        else:\n'
        '            self.running = False\n'
        '            if previous_game == game and turn - previous_turn <= 3:\n'
        '                proposal = result\n'
        '        if not self.running and turn % 8 == 1 and len(jpeg) <= 1_500_000:\n'
        '            self.running = True\n'
        '            Thread(target=self._request, args=(game, turn, brief, jpeg, schemas), daemon=True).start()\n'
        '        return proposal\n'
    ),
    'gaming_agent/policy.py': (
        'import re\n'
        'from dataclasses import dataclass\n'
        '\n'
        'import numpy as np\n'
        '\n'
        'from .model import LocalModel\n'
        'from .protocol import Actions, SKIP, field\n'
        'from .vision import COLORS, analyze, hud_signature, route, thumbnail\n'
        '\n'
        '\n'
        'ARROWS = ("ArrowLeft", "ArrowRight", "ArrowUp", "ArrowDown")\n'
        'WASD = ("a", "d", "w", "s")\n'
        '\n'
        '\n'
        '@dataclass(frozen=True, slots=True)\n'
        'class Brief:\n'
        '    text: str\n'
        '    pointer: str | None\n'
        '    direction: tuple\n'
        '    keys: tuple\n'
        '    player: str | None\n'
        '    target: str | None\n'
        '    dodge: bool\n'
        '    axis: str | None\n'
        '    walls: bool\n'
        '\n'
        '    @classmethod\n'
        '    def parse(cls, info):\n'
        '        inner = field(info, "brief", info)\n'
        '        description = str(field(inner, "description", ""))\n'
        '        controls = str(field(inner, "controls", field(info, "controls", "")))\n'
        '        text = f"{description}\\n{controls}"\n'
        '        lower = text.lower()\n'
        '        input_text = controls.lower()\n'
        '        pointer = ("drag" if re.search(r"\\bdrag\\b", lower)\n'
        '                   else "click" if re.search(r"\\b(click|tap|shoot with (the )?mouse)\\b", lower)\n'
        '                   else "move" if re.search(r"\\b(mouse|cursor|pointer|hover)\\b", lower) else None)\n'
        '        direction = (WASD if re.search(r"\\bwasd\\b|\\bw\\s*[/,]\\s*a\\s*[/,]\\s*s\\s*[/,]\\s*d\\b", input_text)\n'
        '                     else ARROWS if re.search(r"\\barrow|\\bleft\\b.*\\bright\\b|←|→", input_text) else ())\n'
        '        keys = list(direction)\n'
        '        for token in ("Space", "Enter", "Escape", "Shift", "Control", "Tab"):\n'
        '            if re.search(r"\\b" + token.lower() + (r"(?:bar)?\\b" if token == "Space" else r"\\b"), input_text):\n'
        '                keys.append(token)\n'
        '        keys.extend(re.findall(r"\\b(?:press|hold|key)\\s+[`\'\\"\\[]?([a-z0-9])\\b", input_text))\n'
        '        color = "(" + "|".join(COLORS) + ")"\n'
        '        actor = r"(?:player|ship|paddle|avatar|car|character|square|cursor|basket|hero)"\n'
        '        objective = r"(?:target|coin|gem|orb|food|ball|block|item|enemy|hazard|obstacle|dot|star)s?"\n'
        '        player = re.search(r"\\b" + color + r"\\s+" + actor + r"\\b", description.lower())\n'
        '        target = re.search(r"\\b" + color + r"\\s+" + objective + r"\\b", description.lower())\n'
        '        axis = ("x" if re.search(r"\\bhorizontal|left\\s*(?:and|/|or)\\s*right", lower)\n'
        '                else "y" if re.search(r"\\bvertical|up\\s*(?:and|/|or)\\s*down", lower) else None)\n'
        '        return cls(text, pointer, direction, tuple(dict.fromkeys(keys)),\n'
        '                   player[1] if player else None, target[1] if target else None,\n'
        '                   bool(re.search(r"\\b(dodge|avoid|evade)\\b", description.lower())), axis,\n'
        '                   bool(re.search(r"\\b(walls?|maze)\\b", description.lower())))\n'
        '\n'
        '\n'
        'class Policy:\n'
        '    def __init__(self, model=None):\n'
        '        self.model = model if model is not None else LocalModel.from_env()\n'
        '        self.game = 0\n'
        '        self.elapsed = None\n'
        '        self._reset()\n'
        '\n'
        '    def _reset(self):\n'
        '        self.game += 1\n'
        '        self.turn = 0\n'
        '        self.previous = self.hud = self.score = self.last_choice = self.last_target = None\n'
        '        self.feedback = {}\n'
        '        self.brief = None\n'
        '\n'
        '    def _reward(self, info, hud, result):\n'
        '        score = field(info, "score")\n'
        '        if type(score) in (float, int) and type(self.score) in (float, int):\n'
        '            signal = float(np.clip(score - self.score, -1, 1))\n'
        '        elif self.hud is not None and hud is not None and self.hud.shape == hud.shape:\n'
        '            signal = min(0.2, float(np.mean(self.hud != hud)))\n'
        '        else:\n'
        '            signal = 0.0\n'
        '        self.score, self.hud = score, hud\n'
        '        if self.last_choice is None:\n'
        '            return\n'
        '        if (field(result, "accepted") is False or field(result, "ok") is False\n'
        '                or field(result, "status") in ("refused", "rejected", "invalid")):\n'
        '            signal = -1.0\n'
        '        total, count = self.feedback.get(self.last_choice, (0.0, 0))\n'
        '        self.feedback[self.last_choice] = (total + signal, count + 1)\n'
        '\n'
        '    def _choose(self, options):\n'
        '        for action, choice in options:\n'
        '            if action and choice is None:\n'
        '                self.last_choice = None\n'
        '                return action\n'
        '        scored = []\n'
        '        for action, choice in options:\n'
        '            if action:\n'
        '                total, count = self.feedback.get(choice, (0.0, 0))\n'
        '                priority = total / max(count, 1) + 0.45 * np.sqrt(np.log(self.turn + 1) / (count + 1))\n'
        '                scored.append((priority, action, choice))\n'
        '        if scored:\n'
        '            _, action, self.last_choice = max(scored, key=lambda row: row[0])\n'
        '            return action\n'
        '        self.last_choice = None\n'
        '        return SKIP\n'
        '\n'
        '    def _react(self, actions, blobs, image, width, height):\n'
        '        brief = self.brief\n'
        '        blobs = [b for b in blobs if b.y > height * .09]\n'
        '        players = [b for b in blobs if b.color == brief.player]\n'
        '        player = max(players, key=lambda b: b.area, default=None)\n'
        '        targets = [b for b in blobs if b not in players and (not brief.target or b.color == brief.target)]\n'
        '        anchor = (player.x, player.y) if player else (width // 2, height // 2)\n'
        '\n'
        '        def rank(b):\n'
        '            x, y = anchor\n'
        '            distance = abs(b.x - x) / width + abs(b.y - y) / height\n'
        '            repeated = self.last_target is not None and abs(b.x - self.last_target[0]) + abs(b.y - self.last_target[1]) < 12\n'
        '            return 1.5 * b.motion - distance + min(b.area, 200) / 400 - (0.2 if repeated else 0)\n'
        '\n'
        '        target = max(targets, key=rank, default=None)\n'
        '        if brief.pointer == "drag":\n'
        '            return actions.drag(player, target) if player and target else None\n'
        '        if brief.pointer and target:\n'
        '            self.last_target = (target.x, target.y)\n'
        '            return actions.pointer(brief.pointer, target.x, target.y)\n'
        '        if not (brief.direction and player and target):\n'
        '            return None\n'
        '        dx, dy = target.x - player.x, target.y - player.y\n'
        '        if brief.walls and not (brief.axis or brief.dodge):\n'
        '            waypoint = route(image, player, target, width, height)\n'
        '            if waypoint is not None:\n'
        '                dx, dy = waypoint\n'
        '        if brief.dodge:\n'
        '            dx, dy = -dx, -dy\n'
        '            if brief.axis == "x" and (player.x < width * .13 or player.x > width * .87):\n'
        '                dx = width / 2 - player.x\n'
        '            if brief.axis == "y" and (player.y < height * .13 or player.y > height * .87):\n'
        '                dy = height / 2 - player.y\n'
        '        left, right, up, down = brief.direction\n'
        '        choose_x = brief.axis == "x" or (not brief.axis and abs(dx) / width >= abs(dy) / height)\n'
        '        distance = abs(dx) if choose_x else abs(dy)\n'
        '        if distance < 5 and not brief.dodge:\n'
        '            return actions.make("wait", duration_ms=200) or SKIP\n'
        '        key = (left if dx < 0 else right) if choose_x else (up if dy < 0 else down)\n'
        '        return actions.key(key, 100 if distance < 20 else 200)\n'
        '\n'
        '    def step(self, obs, tools):\n'
        '        elapsed = field(obs, "elapsed_s")\n'
        '        if type(elapsed) in (int, float):\n'
        '            if self.elapsed is not None and elapsed < self.elapsed:\n'
        '                self._reset()\n'
        '            self.elapsed = elapsed\n'
        '        info = field(obs, "info", {})\n'
        '        if self.brief is None:\n'
        '            self.brief = Brief.parse(info)\n'
        '        self.turn += 1\n'
        '        jpeg = field(obs, "image")\n'
        '        image = thumbnail(jpeg)\n'
        '        if image is None:\n'
        '            return SKIP\n'
        '        width = field(info, "width", image.shape[1])\n'
        '        height = field(info, "height", image.shape[0])\n'
        '        if type(width) is not int or type(height) is not int or min(width, height) <= 0:\n'
        '            return SKIP\n'
        '        actions = Actions(tools, width, height)\n'
        '        self._reward(info, hud_signature(image), field(obs, "last_action"))\n'
        '        history = field(obs, "history", ())\n'
        '        earlier = thumbnail(field(history[-1], "image", history[-1])) if isinstance(history, (list, tuple)) and history else None\n'
        '        blobs = analyze(image, earlier if earlier is not None else self.previous, width, height)\n'
        '        self.previous = image\n'
        '        reactive = self._react(actions, blobs, image, width, height)\n'
        '        if self.model is not None:\n'
        '            proposal = self.model.poll(self.game, self.turn, self.brief.text, jpeg, actions.specs)\n'
        '            if proposed := actions.accepts(proposal):\n'
        '                return self._choose([(proposed, None)])\n'
        '        if reactive:\n'
        '            return self._choose([(reactive, None)])\n'
        '        keys = self.brief.keys or (() if self.brief.pointer else (*ARROWS, "Enter", "Space"))\n'
        '        choices = [(actions.key(key), ("key", key)) for key in keys]\n'
        '        if self.brief.pointer or not self.brief.keys:\n'
        '            tool = "move" if self.brief.pointer == "drag" else self.brief.pointer or "click"\n'
        '            for x, y in ((.5, .5), (.25, .5), (.75, .5), (.5, .75), (.5, .25)):\n'
        '                choices.append((actions.pointer(tool, int(width * x), int(height * y)), (tool, x, y)))\n'
        '        return self._choose(choices)\n'
    ),
    'gaming_agent/protocol.py': (
        'from collections.abc import Mapping\n'
        '\n'
        '\n'
        'TOOLS = frozenset((\n'
        '    "click", "move", "drag", "scroll", "press_key", "hold_keys", "key_down",\n'
        '    "key_up", "wait", "sequence", "skip",\n'
        '))\n'
        'SKIP = ("skip", {})\n'
        '\n'
        '\n'
        'def field(obj, name, default=None):\n'
        '    if isinstance(obj, Mapping):\n'
        '        return obj.get(name, default)\n'
        '    return getattr(obj, name, default)\n'
        '\n'
        '\n'
        'def _schema(spec):\n'
        '    for name in ("params_schema", "param_schema", "parameters_schema", "input_schema", "json_schema",\n'
        '                 "schema", "parameters", "params"):\n'
        '        value = field(spec, name)\n'
        '        if isinstance(value, Mapping):\n'
        '            return _schema(value) or value\n'
        '    if isinstance(spec, Mapping) and ("properties" in spec or "type" in spec):\n'
        '        return spec\n'
        '    return {}\n'
        '\n'
        '\n'
        'def _valid(value, schema):\n'
        '    if not isinstance(schema, Mapping):\n'
        '        return schema is not False\n'
        '    types = {"object": dict, "array": list, "string": str, "integer": int,\n'
        '             "number": (int, float), "boolean": bool, "null": type(None)}\n'
        '    kind = schema.get("type")\n'
        '    kinds = kind if isinstance(kind, list) else [kind]\n'
        '    if not any(t not in types or (\n'
        '        isinstance(value, types[t]) and not (type(value) is bool and t in ("integer", "number"))\n'
        '    ) for t in kinds):\n'
        '        return False\n'
        '    if "enum" in schema and value not in schema["enum"]:\n'
        '        return False\n'
        '    if "const" in schema and value != schema["const"]:\n'
        '        return False\n'
        '    for keyword, match in (("allOf", all), ("anyOf", any)):\n'
        '        if keyword in schema and not match(_valid(value, s) for s in schema[keyword]):\n'
        '            return False\n'
        '    if "oneOf" in schema and sum(_valid(value, s) for s in schema["oneOf"]) != 1:\n'
        '        return False\n'
        '    if isinstance(value, dict):\n'
        '        props = schema.get("properties", {})\n'
        '        if not all(k in value for k in schema.get("required", ())):\n'
        '            return False\n'
        '        extra = schema.get("additionalProperties", True)\n'
        '        return all(_valid(v, props.get(k, extra)) for k, v in value.items())\n'
        '    if isinstance(value, list):\n'
        '        return (schema.get("minItems", 0) <= len(value) <= schema.get("maxItems", float("inf"))\n'
        '                and all(_valid(v, schema.get("items", {})) for v in value))\n'
        '    if isinstance(value, str):\n'
        '        return schema.get("minLength", 0) <= len(value) <= schema.get("maxLength", float("inf"))\n'
        '    if type(value) in (float, int):\n'
        '        return (schema.get("minimum", -float("inf")) <= value <= schema.get("maximum", float("inf"))\n'
        '                and schema.get("exclusiveMinimum", -float("inf")) < value\n'
        '                < schema.get("exclusiveMaximum", float("inf")))\n'
        '    return True\n'
        '\n'
        '\n'
        'class Actions:\n'
        '    def __init__(self, tools, width, height):\n'
        '        self.specs = {}\n'
        '        self.width, self.height = width, height\n'
        '        for spec in tools:\n'
        '            definition = field(spec, "function", spec)\n'
        '            name = field(definition, "name") or field(definition, "tool")\n'
        '            if name in TOOLS:\n'
        '                self.specs[name] = _schema(definition)\n'
        '\n'
        '    def make(self, tool, **params):\n'
        '        props = self.specs.get(tool, {}).get("properties")\n'
        '        if isinstance(props, Mapping):\n'
        '            params = {k: v for k, v in params.items() if k in props}\n'
        '        return self._validate(tool, params)\n'
        '\n'
        '    def _validate(self, tool, params):\n'
        '        if tool not in self.specs:\n'
        '            return None\n'
        '        spec = self.specs[tool]\n'
        '        for key, value in params.items():\n'
        '            bounds = {\n'
        '                "x": self.width, "y": self.height, "from_x": self.width, "from_y": self.height,\n'
        '                "to_x": self.width, "to_y": self.height, "start_x": self.width, "start_y": self.height,\n'
        '                "end_x": self.width, "end_y": self.height,\n'
        '            }\n'
        '            if key in bounds and (type(value) not in (int, float) or not 0 <= value < bounds[key]):\n'
        '                return None\n'
        '        if tool == "sequence":\n'
        '            steps = params.get("steps", params.get("actions"))\n'
        '            if not isinstance(steps, list) or len(steps) > 200:\n'
        '                return None\n'
        '        if _valid(params, spec):\n'
        '            return tool, params\n'
        '        return None\n'
        '\n'
        '    def pointer(self, tool, x, y):\n'
        '        x, y = max(0, min(self.width - 1, x)), max(0, min(self.height - 1, y))\n'
        '        return self.make(tool, x=x, y=y, button="left") or self.make(tool, x=x, y=y)\n'
        '\n'
        '    def key(self, key, ms=200):\n'
        '        return (self.make("press_key", key=key, duration_ms=ms)\n'
        '                or self.make("press_key", key=key)\n'
        '                or self.make("hold_keys", keys=[key], duration_ms=ms))\n'
        '\n'
        '    def drag(self, start, end):\n'
        '        return self.make("drag", x=start.x, y=start.y, to_x=end.x, to_y=end.y,\n'
        '                         from_x=start.x, from_y=start.y, start_x=start.x, start_y=start.y,\n'
        '                         end_x=end.x, end_y=end.y, duration_ms=200)\n'
        '\n'
        '    def accepts(self, proposal):\n'
        '        if not isinstance(proposal, Mapping):\n'
        '            return None\n'
        '        tool, params = proposal.get("tool"), proposal.get("params")\n'
        '        if isinstance(tool, str) and isinstance(params, dict):\n'
        '            return self._validate(tool, params)\n'
    ),
    'gaming_agent/vision.py': (
        'from collections import deque\n'
        'from dataclasses import dataclass\n'
        'from io import BytesIO\n'
        '\n'
        'import numpy as np\n'
        'from PIL import Image\n'
        '\n'
        '\n'
        'COLORS = ("red", "green", "blue", "yellow", "orange", "purple", "cyan", "white")\n'
        '\n'
        '\n'
        '@dataclass(frozen=True, slots=True)\n'
        'class Blob:\n'
        '    x: int\n'
        '    y: int\n'
        '    color: str\n'
        '    area: int\n'
        '    motion: float\n'
        '\n'
        '\n'
        'def thumbnail(jpeg, size=112):\n'
        '    try:\n'
        '        with Image.open(BytesIO(jpeg)) as image:\n'
        '            if image.width * image.height > 8_400_000:\n'
        '                return None\n'
        '            image.draft("RGB", (size * 2, size * 2))\n'
        '            image.thumbnail((size, size), Image.Resampling.BILINEAR)\n'
        '            return np.asarray(image.convert("RGB"))\n'
        '    except (OSError, TypeError, ValueError, Image.DecompressionBombError):\n'
        '        return None\n'
        '\n'
        '\n'
        'def _labels(image):\n'
        '    rgb = image.astype(np.float32)\n'
        '    red, green, blue = rgb.transpose(2, 0, 1)\n'
        '    high, low = rgb.max(axis=2), rgb.min(axis=2)\n'
        '    chroma = high - low\n'
        '    delta = np.maximum(chroma, 1)\n'
        '    hue = np.where(high == red, (green - blue) / delta,\n'
        '                   np.where(high == green, 2 + (blue - red) / delta, 4 + (red - green) / delta))\n'
        '    hue %= 6\n'
        '    label = np.select((hue >= 5.5, hue >= 4.45, hue >= 3.45, hue >= 2.65,\n'
        '                       hue >= 1.5, hue >= 0.75, hue >= 0.25), (1, 6, 3, 7, 2, 4, 5), 1)\n'
        '    colorful = (high >= 80) & (chroma >= 45) & (chroma >= high * 0.38)\n'
        '    return np.where(colorful, label, np.where((low > 185) & (chroma < 40), 8, 0))\n'
        '\n'
        '\n'
        'def analyze(image, previous, width, height):\n'
        '    labels = _labels(image)\n'
        '    h, w = labels.shape\n'
        '    motion = np.zeros((h, w), dtype=bool)\n'
        '    if previous is not None and previous.shape == image.shape:\n'
        '        motion = np.max(np.abs(image.astype(np.int16) - previous), axis=2) > 28\n'
        '    # Flat indices with a sentinel border keep component traversal independent of edge checks.\n'
        '    board = np.pad(labels, 1).ravel().tolist()\n'
        '    stride = w + 2\n'
        '    moved = np.pad(motion, 1).ravel().tolist()\n'
        '    blobs = []\n'
        '    for seed, label in enumerate(board):\n'
        '        if not label:\n'
        '            continue\n'
        '        board[seed] = 0\n'
        '        stack = [seed]\n'
        '        n = sx = sy = nmoved = 0\n'
        '        while stack:\n'
        '            i = stack.pop()\n'
        '            y, x = divmod(i, stride)\n'
        '            n += 1\n'
        '            sx += x - 1\n'
        '            sy += y - 1\n'
        '            nmoved += moved[i]\n'
        '            for j in (i - 1, i + 1, i - stride, i + stride):\n'
        '                if board[j] == label:\n'
        '                    board[j] = 0\n'
        '                    stack.append(j)\n'
        '        if 3 <= n <= min(w * h * 0.18, 1600):\n'
        '            blobs.append(Blob(int((sx / n + .5) * width / w),\n'
        '                              int((sy / n + .5) * height / h),\n'
        '                              COLORS[label - 1], n, nmoved / n))\n'
        '    return blobs\n'
        '\n'
        '\n'
        'def hud_signature(image):\n'
        '    if image is not None:\n'
        '        h, w = image.shape[:2]\n'
        '        return (image[:max(2, h // 10), :max(2, w // 3)].astype(np.int16) // 48)\n'
        '\n'
        '\n'
        'def route(image, player, target, width, height):\n'
        '    h, w = image.shape[:2]\n'
        '    step = max(1, max(h, w) // 28)\n'
        '    rgb = image[::step, ::step].astype(np.int16)\n'
        '    rows, cols = rgb.shape[:2]\n'
        '    background = np.median(rgb.reshape(-1, 3), axis=0)\n'
        '    raw = (np.ptp(rgb, axis=2) < 50) & (np.max(np.abs(rgb - background), axis=2) > 33)\n'
        '    padded = np.pad(raw, 1)\n'
        '    raw &= (padded[:-2, 1:-1] | padded[2:, 1:-1] | padded[1:-1, :-2] | padded[1:-1, 2:])\n'
        '    raw[:max(1, rows // 10)] = True\n'
        '    if not raw[max(1, rows // 10):].any():\n'
        '        return None\n'
        '    buffer = np.pad(raw, 1)\n'
        '    blocked = np.logical_or.reduce([buffer[y:y + rows, x:x + cols]\n'
        '                                    for y in range(3) for x in range(3)])\n'
        '    def cell(blob):\n'
        '        return (min(cols - 1, round(blob.x * w / width / step)),\n'
        '                min(rows - 1, round(blob.y * h / height / step)))\n'
        '\n'
        '    start, end = cell(player), cell(target)\n'
        '\n'
        '    def clear(a, b):\n'
        '        ax, ay = a\n'
        '        bx, by = b\n'
        '        if ay == by:\n'
        '            cells = blocked[ay, min(ax, bx):max(ax, bx) + 1]\n'
        '        else:\n'
        '            cells = blocked[min(ay, by):max(ay, by) + 1, ax]\n'
        '        return not cells.any()\n'
        '\n'
        '    sx, sy = start\n'
        '    ex, ey = end\n'
        '    dx, dy = target.x - player.x, target.y - player.y\n'
        '    horizontal = abs(dx) / width >= abs(dy) / height\n'
        '    x_first, y_first = (ex, sy), (sx, ey)\n'
        '    first, second = (x_first, y_first) if horizontal else (y_first, x_first)\n'
        '    if clear(start, first) and clear(first, end):\n'
        '        return None\n'
        '    if second != start and clear(start, second) and clear(second, end):\n'
        '        return (0, dy) if horizontal else (dx, 0)\n'
        '\n'
        '    queue = deque([start])\n'
        '    visited = {start: None}\n'
        '    while queue:\n'
        '        x, y = queue.popleft()\n'
        '        if (x, y) == end:\n'
        '            return visited[end]\n'
        '        neighbors = [(x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)]\n'
        '        neighbors.sort(key=lambda point: abs(point[0] - end[0]) + abs(point[1] - end[1]))\n'
        '        for nx, ny in neighbors:\n'
        '            if (nx, ny) not in visited and 0 <= nx < cols and 0 <= ny < rows and not blocked[ny, nx]:\n'
        '                visited[nx, ny] = (visited[x, y] or ((nx - x) * width / cols, (ny - y) * height / rows))\n'
        '                queue.append((nx, ny))\n'
    ),
}
for relative, source in SOURCES.items():
    path = WORK / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding="utf-8")

In [ ]:
if MODEL_DIR is not None:
    import importlib.util
    import subprocess
    import time
    from urllib.request import ProxyHandler, build_opener

    model = Path(MODEL_DIR).resolve()
    if (not model.is_relative_to(INPUT.resolve()) or not model.is_dir()
        or importlib.util.find_spec("vllm") is None):
        print("Attached model directory or vLLM missing; using the CPU policy.")
    else:
        port, name = "8765", "gga-local"
        with (WORK / "model-server.log").open("w") as log:
            process = subprocess.Popen([
                sys.executable, "-m", "vllm.entrypoints.openai.api_server",
                "--model", str(model), "--served-model-name", name,
                "--host", "127.0.0.1", "--port", port,
                "--max-model-len", "4096", "--max-num-seqs", "1",
            ], stdout=log, stderr=subprocess.STDOUT)
        deadline = time.monotonic() + 180
        opener = build_opener(ProxyHandler({}))
        while process.poll() is None and time.monotonic() < deadline:
            try:
                with opener.open(f"http://127.0.0.1:{port}/health", timeout=1):
                    os.environ.update(GGA_MODEL_NAME=name, GGA_MODEL_PORT=port)
                break
            except OSError:
                time.sleep(1)
        else:
            if process.poll() is None:
                process.kill()
                process.wait()
            print("Model server did not start; using the CPU policy. See model-server.log.")

In [ ]:
from kaggle_evaluation.gga.gga_inference_server import GGAInferenceServer

server = GGAInferenceServer()
server.run()